# Aplicando PCA

## Redução de dimensionalidade utilizando PCA

Nesta etapa é aplicada uma técnica de redução de dimensionalidade baseada em Análise de Componentes Principais (PCA - Principal Component Analysis). O objetivo é transformar o conjunto original de variáveis municipais agregadas em um novo espaço de menor dimensão, preservando a maior quantidade possível de informação presente nos dados.

A aplicação do PCA permite reduzir a complexidade da representação dos candidatos, substituindo diversas variáveis correlacionadas por um conjunto menor de componentes principais. Esses componentes são combinações lineares das variáveis originais e representam os principais padrões de variação existentes nos dados.

Antes da aplicação do PCA, são removidas variáveis relacionadas diretamente ao desempenho eleitoral dos candidatos, como quantidade de votos recebidos e número de municípios onde houve votação. Essa remoção evita o problema de vazamento de informação (*data leakage*), no qual variáveis diretamente relacionadas ao resultado final poderiam influenciar indevidamente análises ou modelos posteriores.

São definidos também os arquivos de entrada e saída e o percentual mínimo de variância que deve ser preservado pelos componentes principais.

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

INPUT_FILE = "candidate_representation_table.csv"

OUTPUT_FILE = "candidate_representation_pca.csv"

VARIANCE_TO_KEEP = 0.90

## Preparação da base para aplicação do PCA

Inicialmente é carregada a tabela de representação dos candidatos gerada na etapa anterior do processamento. Essa tabela contém informações de identificação dos candidatos, características individuais, variáveis municipais agregadas e métricas relacionadas ao desempenho eleitoral.

Para preparar os dados para o PCA, as variáveis são divididas em dois grupos. O primeiro grupo corresponde aos metadados, que incluem informações como cargo, partido, características demográficas e a variável objetivo de sucesso eleitoral. Essas informações são mantidas separadas porque não representam dimensões contínuas adequadas para a transformação pelo PCA.

O segundo grupo corresponde às variáveis candidatas a entrar na redução de dimensionalidade. Além dos metadados, também são removidas as variáveis relacionadas ao desempenho eleitoral, pois elas poderiam carregar informações diretamente associadas ao resultado da eleição e introduzir viés nas análises posteriores.

In [2]:
df = pd.read_csv(INPUT_FILE)

print("Rows:", len(df))
print("Columns:", len(df.columns))

metadata_cols = [

    "Cargo",
    "Nome candidato",
    "Partido",
    "Turno",

    "Cor/raça",
    "Estado civil",
    "Faixa etária",
    "Gênero",
    "Grau de instrução",
    "Ocupação",

    "Situação totalização",
    "UF",
    "Região",

    "success"
]

metadata_cols = [
    c
    for c in metadata_cols
    if c in df.columns
]

leakage_cols = [

    "total_vote_share",
    "mean_vote_share",
    "max_vote_share",

    "total_nominal_votes",
    "total_valid_votes",

    "n_municipalities"
]

leakage_cols = [
    c
    for c in leakage_cols
    if c in df.columns
]

print("\nRemoved leakage variables:")
print(leakage_cols)

Rows: 25186
Columns: 47

Removed leakage variables:
['total_vote_share', 'mean_vote_share', 'max_vote_share', 'total_nominal_votes', 'total_valid_votes', 'n_municipalities']


## Seleção das variáveis utilizadas no PCA e normalização dos dados

Nesta etapa são selecionadas apenas as variáveis numéricas que representam características dos municípios associados aos candidatos. Variáveis identificadoras, categóricas e aquelas relacionadas ao desempenho eleitoral são excluídas, permanecendo apenas atributos adequados para operações matemáticas do PCA.

Após a seleção das variáveis, valores ausentes são substituídos por zero para garantir que a matriz de entrada esteja completa. Em seguida, os dados são padronizados utilizando o método StandardScaler.

A padronização é necessária porque as variáveis possuem diferentes escalas e unidades de medida. Sem esse processo, atributos com valores numericamente maiores poderiam dominar a construção dos componentes principais, reduzindo a influência de variáveis com escalas menores.

In [3]:
numeric_cols = []

for col in df.columns:

    if col in metadata_cols:
        continue

    if col in leakage_cols:
        continue

    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_cols.append(col)

print("\nNumeric columns used in PCA:")
print(len(numeric_cols))

X = df[numeric_cols].copy()

X = X.fillna(0)

print("\nStandardizing...")

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)



Numeric columns used in PCA:
27

Standardizing...


## Transformação dos dados utilizando PCA

Após a padronização dos dados, é aplicada a Análise de Componentes Principais. O algoritmo identifica novas direções no espaço dos dados que maximizam a variância explicada, criando componentes que representam combinações das variáveis originais.

O número de componentes principais é definido automaticamente com base no percentual de variância que deve ser preservado. Neste caso, são mantidos componentes suficientes para explicar 90% da variabilidade presente nos dados originais.

A matriz resultante do PCA é transformada em uma nova tabela, onde cada coluna representa um componente principal. Esses novos atributos substituem o grande conjunto inicial de variáveis numéricas, mantendo os padrões mais relevantes dos dados em uma representação de menor dimensão.

Ao final, os componentes principais são combinados novamente com os metadados dos candidatos, preservando informações como cargo, partido, características individuais e variável de sucesso eleitoral.

In [4]:
print("Running PCA...")

pca = PCA(n_components=VARIANCE_TO_KEEP,random_state=42)

X_pca = pca.fit_transform(X_scaled)

print("\nNumber of PCA components:",X_pca.shape[1])

print("Explained variance:",round(pca.explained_variance_ratio_.sum() * 100,2),"%")

pca_cols = [
    f"PC{i+1}"
    for i in range(X_pca.shape[1])
]

pca_df = pd.DataFrame(X_pca,columns=pca_cols)

final_df = pd.concat([df[metadata_cols].reset_index(drop=True),pca_df],axis=1)

print("\nFinal shape:")
print(final_df.shape)

final_df.to_csv(OUTPUT_FILE,index=False)

print("\nSaved:",OUTPUT_FILE)

Running PCA...

Number of PCA components: 8
Explained variance: 90.45 %

Final shape:
(25186, 22)

Saved: candidate_representation_pca.csv


## Interpretação dos componentes principais

Após a transformação dos dados, são analisados os *loadings* do PCA. Os loadings representam a contribuição de cada variável original para a formação de cada componente principal.

Essa análise permite identificar quais variáveis municipais possuem maior influência em cada componente, ajudando a interpretar o significado dos novos atributos criados pelo PCA. Para cada componente, são selecionadas as variáveis com maior peso absoluto, indicando quais características mais contribuem para a variação representada.

Por fim, a matriz de loadings é exportada para um arquivo separado, permitindo análises posteriores sobre a relação entre os componentes principais e as variáveis originais.

In [5]:
loadings = pd.DataFrame(pca.components_.T,columns=pca_cols,index=numeric_cols)

print("\n========================================")
print("TOP VARIABLES PER COMPONENT")
print("========================================")

for pc in pca_cols[:10]:

    print(f"\n{pc}")

    top_vars = (loadings[pc].abs().sort_values(ascending=False).head(10))

    print(top_vars)

loadings.to_csv("pca_loadings.csv")

print("\nSaved: pca_loadings.csv")

print("\nDone.")


TOP VARIABLES PER COMPONENT

PC1
População no último censo                        0.259883
Pessoal ocupado em postos de trabalho formais    0.255029
Número de estabelecimentos de ensino médio       0.254236
Matrículas no ensino médio                       0.253136
Matrículas no ensino fundamental                 0.252966
Docentes no ensino fundamental                   0.251492
Total de despesas brutas empenhadas              0.250328
Total de receitas brutas realizadas              0.250201
Docentes no ensino médio                         0.247310
Estabelecimentos de Saúde SUS                    0.237991
Name: PC1, dtype: float64

PC2
Percentual da população com rendimento nominal mensal per capita de até 1/2 salário mínimo    0.356960
IDEB – Anos iniciais do ensino fundamental (Rede pública)                                     0.348809
Índice de Desenvolvimento Humano Municipal (IDHM)                                             0.300038
IDEB – Anos finais do ensino fundamental (Rede

# Subgroup Discovery

## Descoberta de subgrupos utilizando a representação PCA

Nesta etapa é aplicada a técnica de Subgroup Discovery sobre a representação reduzida dos candidatos obtida anteriormente utilizando PCA. O objetivo é identificar subconjuntos de candidatos que apresentam uma taxa de eleição diferente da observada no conjunto completo de dados.

A descoberta de subgrupos busca encontrar padrões descritivos nos dados, identificando combinações de características que caracterizam grupos onde a ocorrência do evento de interesse é significativamente maior ou menor que a média geral. Neste caso, o evento analisado é o sucesso eleitoral, representado pela variável binária indicando se o candidato foi eleito.

A busca é realizada utilizando a biblioteca PySubgroup, que permite explorar combinações de condições sobre as variáveis disponíveis. São definidos parâmetros que controlam a quantidade de resultados retornados, a profundidade máxima das regras encontradas e a largura da busca em feixe (*beam search*).

A entrada utilizada corresponde à tabela contendo os componentes principais gerados pelo PCA, combinados com informações de identificação dos candidatos.

In [6]:
import pandas as pd
import pysubgroup as ps

INPUT_FILE = "candidate_representation_pca.csv"

OUTPUT_FILE = "subgroup_pca_results.csv"

TOP_K = 100

MAX_DEPTH = 3

BEAM_WIDTH = 200

## Preparação da base para descoberta de subgrupos

Inicialmente a tabela resultante do PCA é carregada e são verificadas suas dimensões para confirmar a quantidade de candidatos e atributos disponíveis.

Em seguida, é criada a variável objetivo utilizada na busca de padrões. A variável `elected` representa se o candidato foi eleito, sendo criada a partir da situação final da candidatura. Essa variável será utilizada pelo algoritmo como alvo da descoberta de subgrupos.

Antes da execução da busca, é realizado um novo processo de remoção de variáveis que poderiam causar vazamento de informação (*data leakage*). São removidas variáveis diretamente relacionadas ao resultado eleitoral, como quantidade de votos, participação eleitoral e situação final da candidatura. Também são removidas variáveis identificadoras ou altamente específicas dos candidatos, como nome, cargo e turno, pois o objetivo é encontrar padrões associados às características representadas pelo PCA e não memorizar indivíduos específicos.

Após a remoção, permanece apenas o conjunto de atributos utilizado para procurar padrões explicativos relacionados ao sucesso eleitoral.

In [7]:
df = pd.read_csv(INPUT_FILE)

print("Rows:", len(df))
print("Columns:", len(df.columns))

df["elected"] = (df["Situação totalização"]== "Eleito")

print("\nElection rate:")

print(round(100 * df["elected"].mean(),2),"%")

remove_cols = [

    # identifiers
    "Nome candidato",
    "Cargo",
    "Turno",

    # target source
    "Situação totalização",

    # target
    "elected",
    "success",

    # electoral performance
    "total_nominal_votes",
    "total_valid_votes",
    "total_vote_share",
    "mean_vote_share",
    "max_vote_share",

    # usually remove
    "n_municipalities",
    "Ocupação",
    "Cor/raça",
    "Estado civil",
    "Faixa etária",
    "Gênero",
    "Grau de instrução",
    "Ocupação"
]

remove_cols = [
    c
    for c in remove_cols
    if c in df.columns
]

search_df = df.drop(columns=remove_cols,errors="ignore")


for col in search_df.columns:

    if col.startswith("PC"):
        continue

    if search_df[col].dtype == "object":
        continue

    if search_df[col].nunique() <= 10:

        search_df[col] = (search_df[col].astype(str))


Rows: 25186
Columns: 22

Election rate:
6.46 %


## Definição do espaço de busca e execução do Subgroup Discovery

Nesta etapa são construídos os possíveis seletores utilizados pelo algoritmo de descoberta de subgrupos. Cada seletor representa uma condição que pode ser combinada com outras para formar uma regra descritiva.

Como os componentes principais do PCA são variáveis contínuas, eles são mantidos como atributos numéricos. Variáveis numéricas com poucos valores distintos são convertidas para categorias, pois podem representar atributos discretos e facilitar a criação de regras interpretáveis.

O alvo da busca é definido como a ocorrência do evento "eleito". O algoritmo procura subconjuntos onde a proporção de candidatos eleitos seja diferente da proporção encontrada no conjunto completo.

A função de qualidade utilizada é a WRAcc (*Weighted Relative Accuracy*), que avalia simultaneamente o tamanho do grupo encontrado e o quanto ele se diferencia da distribuição geral do alvo. Dessa forma, grupos muito pequenos ou pouco informativos recebem menor prioridade.

A busca é realizada utilizando o algoritmo Beam Search, que explora diferentes combinações de seletores mantendo apenas os candidatos mais promissores a cada etapa. A profundidade máxima controla o número de condições que podem compor cada subgrupo.

In [8]:
search_df["elected"] = df["elected"]

feature_df = search_df.drop(columns=["elected"])

search_space = ps.create_selectors(feature_df)

print("\nNumber of selectors:",len(search_space))

target = ps.BinaryTarget("elected",True)

qf = ps.WRAccQF()

task = ps.SubgroupDiscoveryTask(data=search_df,target=target,search_space=search_space,result_set_size=TOP_K,depth=MAX_DEPTH,qf=qf)

print("\nRunning subgroup discovery...")

result = ps.BeamSearch(beam_width=BEAM_WIDTH).execute(task)



Number of selectors: 104

Running subgroup discovery...


d:\Users\CLIENTE\anaconda3\envs\desc_env\Lib\site-packages\pysubgroup\binary_target.py:356: RuntimeWarning: invalid value encountered in divide
  p_subgroup = np.divide(positives_subgroup, instances_subgroup)


## Processamento e armazenamento dos subgrupos encontrados

Após a execução do algoritmo, os subgrupos descobertos são convertidos para uma tabela estruturada. Cada linha representa uma regra encontrada pelo algoritmo e contém informações sobre sua qualidade e características.

Entre as métricas analisadas estão a quantidade de candidatos pertencentes ao subgrupo, o número de candidatos eleitos dentro dele, a proporção de sucesso eleitoral no grupo e o ganho relativo em relação ao conjunto completo.

Os resultados são exportados para um arquivo CSV, permitindo análises posteriores, comparação entre padrões encontrados ou visualização das regras descobertas.

In [9]:

result_df = result.to_dataframe()

print("\nSubgroups found:",len(result_df))

result_df.to_csv(OUTPUT_FILE,index=False
)

print("\nSaved:",OUTPUT_FILE)


Subgroups found: 100

Saved: subgroup_pca_results.csv


## Análise dos principais subgrupos encontrados

Nesta etapa são exibidos os melhores subgrupos identificados pelo algoritmo de acordo com a função de qualidade utilizada.

São apresentadas métricas como a qualidade da regra, a descrição do subgrupo, seu tamanho, quantidade de candidatos eleitos, proporção de sucesso dentro do grupo e o aumento relativo (*lift*) em comparação com a taxa geral de eleição.

Essa análise permite interpretar quais combinações de características presentes na representação PCA estão associadas a diferentes padrões de sucesso eleitoral. Os resultados devem ser interpretados como associações encontradas nos dados e não como relações causais.

In [10]:
print("\n==============================")
print("TOP PCA SUBGROUPS")
print("==============================")

cols = [
    c
    for c in [
        "quality",
        "subgroup",
        "size_sg",
        "positives_sg",
        "target_share_sg",
        "target_share_dataset",
        "lift"
    ]
    if c in result_df.columns
]

print(result_df[cols].head(30))

print("\nDone.")


TOP PCA SUBGROUPS
     quality                                        subgroup  size_sg  \
0   0.014397                                       PC1<-2.50     5037   
1   0.009590                PC1<-2.50 AND Região=='NORDESTE'     2809   
2   0.009349                         PC1<-2.50 AND PC5>=0.78     2996   
3   0.009117                                       PC6<-0.65     5037   
4   0.008208                         PC1<-2.50 AND PC2>=1.75     2775   
5   0.007248                                       PC5>=0.78     5038   
6   0.007186  PC1<-2.50 AND PC5>=0.78 AND Região=='NORDESTE'     2059   
7   0.006529                PC5>=0.78 AND Região=='NORDESTE'     2408   
8   0.005880                         PC1<-2.50 AND PC3<-0.79     1330   
9   0.005856                         PC1<-2.50 AND PC6<-0.65     1788   
10  0.005662           PC1<-2.50 AND PC2>=1.75 AND PC5>=0.78     1415   
11  0.005617                                   Partido=='PL'     1479   
12  0.005205  PC1<-2.50 AND PC3<